# Large images: a 2 GiB public OME-Zarr, read where it lives

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/LargeImages/LargeImages_demo.ipynb)

An ExaSPIM light-sheet volume from the Allen Institute for Neural Dynamics (specimen 822174, CC BY 4.0): 513 × 1331 × 1775 uint16 voxels at level 0, 2.4 GB uncompressed, in 256³ chunks, four pyramid levels, on a public S3 bucket. Nothing is downloaded whole: KonfAI reads the chunks a request touches and no more, and a TRANSFORM over it streams region by region under a memory budget.

`FSSPEC_S3_ANON=true` reads the public bucket without an AWS account; `konfai[s3]` carries the filesystem.


In [ ]:
# Setup: find KonfAI (cloning it on Colab), install what is missing, load the notebook helpers.
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples").is_dir())
sys.path.insert(0, str(REPO_DIR / "examples"))

from konfai_demo import setup, show

EXAMPLE_DIR, DATASET_DIR, DEVICE = setup(REPO_DIR, "LargeImages", ("konfai", f"{REPO_DIR}[imaging,s3]"), "matplotlib")
os.environ["FSSPEC_S3_ANON"] = "true"  # a public bucket: no credentials
os.chdir(EXAMPLE_DIR)


## 1. The store, by its metadata

`get_infos` reads the pyramid's metadata and nothing else: the shape and geometry of each level, without touching a chunk. The store is named the way a `dataset_filenames` entry names it, `<uri>:omezarr`, and a pyramid level is selected with `@<level>`.


In [ ]:
from konfai.utils.ome_zarr import get_ome_zarr_info, read_ome_zarr_data_slice

STORE = "s3://aind-open-data/exaSPIM_822174_2026-04-28_12-29-55_processed_2026-07-09_03-49-09/fusion2halves/SPIM.ome.zarr"
for level in range(4):
    info = get_ome_zarr_info(STORE, level=level)
    print(f"level {level}: axes {info['axes']}, shape {info['shape']}, chunks {info.get('chunks')}")


## 2. A coarse overview, then one native window

Level 3 is the whole volume at an eighth of the resolution: 4.7 MB, read whole as an overview. From it, pick a window and read the same place at level 0, native resolution: one 256² plane of one chunk, 128 KiB materialised out of 2.4 GB.


In [ ]:
import numpy as np

overview, _ = read_ome_zarr_data_slice(STORE, (slice(0, 1), slice(0, 64), slice(0, 166), slice(0, 221)), level=3)
overview = overview[0]
z = int(np.argmax(overview.reshape(overview.shape[0], -1).mean(axis=1)))  # the brightest plane
y, x = (np.array(np.unravel_index(int(np.argmax(overview[z])), overview[z].shape)) * 8).tolist()
y0, x0 = max(0, min(y - 128, 1331 - 256)), max(0, min(x - 128, 1775 - 256))
window, _ = read_ome_zarr_data_slice(STORE, (slice(0, 1), slice(z * 8, z * 8 + 1), slice(y0, y0 + 256), slice(x0, x0 + 256)), level=0)
print("overview", overview.shape, "| native window", window[0, 0].shape, "at z", z * 8, "y", y0, "x", x0, "|", window.nbytes // 1024, "KiB materialised")
show([(f"level 3, plane {z}", overview[z], "gray"), (f"level 0, 256² window at z={z * 8}", window[0, 0], "gray")])


## 3. A TRANSFORM that streams it

A dataset root holds `<case>/<group>.ome.zarr`: here the processed asset is the root, `fusion2halves` the case and `SPIM` the group. This chain casts the uint16 voxels to float (torch has no kernels for unsigned 16-bit payloads, and the plan says so), clips the intensities and writes the level-2 volume (128 × 332 × 443) to a local HDF5 under a 256 MiB budget: the plan prints the verdict (`STREAM`), the sweep cuts regions along the chunk grid and grows them as the measured hold allows, and the run's last line says what it held. Level 0 is the same command with `@0` and a longer wait on the network.


In [ ]:
import konfai

ROOT = STORE.rsplit("/", 2)[0]  # the processed asset: <root>/<case>/<group>.ome.zarr
from konfai.data.transform import Clip, TensorCast, Write

result = konfai.transform(
    "EXASPIM_L2",
    ROOT + ":omezarr@2",  # the dataset root is the asset: case `fusion2halves`, group `SPIM` (SPIM.ome.zarr)
    {"SPIM": {"SPIM": [TensorCast(dtype="float32"), Clip(min_value=0.0, max_value=20000.0), Write(dataset="./Out:h5")]}},
    memory_budget="256MiB",
    transforms_dir=EXAMPLE_DIR / "Transforms",
    overwrite=True,
)
print((EXAMPLE_DIR / "Transforms" / "EXASPIM_L2" / "log_0.txt").read_text().splitlines()[-1])


## What just happened

- No chunk was read twice, and no volume whole: the overview cost 4.7 MB, the native window one chunk, the transform one pass over level 2 in regions sized against the budget.
- The same `dataset_filenames` entry works in a `Config.yml` for training on the store, patch by patch: {doc}`../usage/large-images` covers the chunk shapes and patch sizes that make it fast.
